Install Packages

In [ ]:
!pip install -q transformers datasets evaluate accelerate

Mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

Imports and device

In [ ]:
import math
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

Set Seed

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Load SST-2 dataset

In [ ]:
dataset = load_dataset("glue", "sst2")
print(dataset)
print(dataset["train"][0])

 Load tokenizer

In [ ]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

Tokenize dataset

In [ ]:
max_seq_length = 512

def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=max_seq_length,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["sentence", "idx"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset["train"][0])

Build dataloaders

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataloader = DataLoader(
    tokenized_dataset["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)

eval_dataloader = DataLoader(
    tokenized_dataset["validation"],
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator,
)

Load baseline model first

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.to(device)
print(type(model))

Inspect RoBERTa attention module names

In [ ]:
for name, module in model.named_modules():
    if "attention" in name and isinstance(module, nn.Linear):
        print(name)

Define LoRALinear

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()

        if not isinstance(base_linear, nn.Linear):
            raise TypeError("base_linear must be nn.Linear")

        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r if r > 0 else 1.0

        # Keep the original frozen weight and bias
        self.weight = base_linear.weight
        self.weight.requires_grad = False

        if base_linear.bias is not None:
            self.bias = base_linear.bias
            self.bias.requires_grad = False
        else:
            self.bias = None

        self.dropout = nn.Dropout(dropout)

        if r > 0:
            # A: (r, in_features)
            # B: (out_features, r)
            self.lora_A = nn.Parameter(torch.zeros(r, self.in_features))
            self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))

            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)
        else:
            self.lora_A = None
            self.lora_B = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_output = F.linear(x, self.weight, self.bias)

        if self.r > 0:
            lora_output = self.dropout(x) @ self.lora_A.t()
            lora_output = lora_output @ self.lora_B.t()
            base_output = base_output + self.scaling * lora_output

        return base_output

Freeze all parameters first

In [ ]:
for param in model.parameters():
    param.requires_grad = False

Replace query/value with LoRA layers

In [ ]:
def replace_query_value_with_lora(model: nn.Module, r: int = 8, alpha: int = 8, dropout: float = 0.0):
    replaced = []

    for layer_idx, layer in enumerate(model.roberta.encoder.layer):
        old_query = layer.attention.self.query
        layer.attention.self.query = LoRALinear(old_query, r=r, alpha=alpha, dropout=dropout)
        replaced.append(f"layer {layer_idx} query")

        old_value = layer.attention.self.value
        layer.attention.self.value = LoRALinear(old_value, r=r, alpha=alpha, dropout=dropout)
        replaced.append(f"layer {layer_idx} value")

    return replaced

replaced_layers = replace_query_value_with_lora(model, r=8, alpha=8, dropout=0.0)
print(f"Replaced {len(replaced_layers)} layers.")
print(replaced_layers[:5], "...")

Unfreeze classification head

In [ ]:
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

Check trainable parameters

In [ ]:
def print_trainable_parameters(model: nn.Module):
    total_params = 0
    trainable_params = 0

    for _, param in model.named_parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    ratio = 100 * trainable_params / total_params
    print(f"Trainable params: {trainable_params}")
    print(f"Total params:     {total_params}")
    print(f"Trainable ratio:  {ratio:.4f}%")

print_trainable_parameters(model)

NameError: name 'model' is not defined

Print trainable parameter names

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

Move model to device again

In [ ]:
model.to(device)

Define optimizer, scheduler, metric

In [ ]:
learning_rate = 5e-4
num_epochs = 60   # paper uses 60 for SST-2

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate
)

num_training_steps = num_epochs * len(train_dataloader)
num_warmup_steps = int(0.06 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

metric = evaluate.load("glue", "sst2")

Quick forward sanity check

In [ ]:
batch = next(iter(train_dataloader))
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.loss)
print(outputs.logits.shape)

Training function

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    return avg_loss

Evaluation function

In [ ]:
def evaluate_model(model, dataloader, metric, device):
    model.eval()
    metric = evaluate.load("glue", "sst2")

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        metric.add_batch(
            predictions=predictions.cpu(),
            references=batch["labels"].cpu()
        )

    return metric.compute()

Run training

In [ ]:
for epoch in range(num_epochs):
    avg_train_loss = train_one_epoch(model, train_dataloader, optimizer, scheduler, device)
    eval_result = evaluate_model(model, eval_dataloader, metric, device)

    print(f"Epoch {epoch + 1}")
    print(f"Average train loss: {avg_train_loss:.4f}")
    print(f"Eval result: {eval_result}")

Optional GPU memory check

In [ ]:
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

Save model checkpoint

In [ ]:
import os

save_dir = "/content/drive/MyDrive/roberta_lora_sst2_base"
os.makedirs(save_dir, exist_ok=True)

lora_state_dict = {
    name: param.cpu()
    for name, param in model.state_dict().items()
    if "lora_" in name or "classifier" in name
}

torch.save(lora_state_dict, os.path.join(save_dir, "lora_classifier.pt"))
tokenizer.save_pretrained(save_dir)

print(f"Saved LoRA + classifier weights to {save_dir}")

MRPC & RoBERTa

In [ ]:
# 1. Load MRPC
mrpc_dataset = load_dataset("glue", "mrpc")

# 2. Tokenize MRPC (Notice we pass both sentence1 and sentence2)
def tokenize_mrpc(examples):
    return tokenizer(examples["sentence1"], examples["sentence2"], truncation=True)

tokenized_mrpc = mrpc_dataset.map(tokenize_mrpc, batched=True)
tokenized_mrpc = tokenized_mrpc.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_mrpc = tokenized_mrpc.rename_column("label", "labels")
tokenized_mrpc.set_format("torch")

# 3. Create DataLoaders
mrpc_train_dataloader = DataLoader(
    tokenized_mrpc["train"], shuffle=True, batch_size=16, collate_fn=data_collator
)
mrpc_eval_dataloader = DataLoader(
    tokenized_mrpc["validation"], shuffle=False, batch_size=32, collate_fn=data_collator
)

In [ ]:
# 1. Load a fresh baseline model so we don't bleed SST-2 weights into this run
mrpc_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 2. Freeze all base parameters
for param in mrpc_model.parameters():
    param.requires_grad = False

# 3. Apply LoRA using your previously defined function
replace_query_value_with_lora(mrpc_model, r=8, alpha=8, dropout=0.05)

# 4. Unfreeze the classification head
for name, param in mrpc_model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

mrpc_model.to(device)

print("MRPC Model Parameters:")
print_trainable_parameters(mrpc_model)

In [ ]:
mrpc_epochs = 30

# 1. New Optimizer
mrpc_optimizer = torch.optim.AdamW(
    [p for p in mrpc_model.parameters() if p.requires_grad],
    lr=4e-4
)

# 2. Calculate steps for 30 epochs
mrpc_num_training_steps = mrpc_epochs * len(mrpc_train_dataloader)
mrpc_num_warmup_steps = int(0.06 * mrpc_num_training_steps) # 6% Warmup

# 3. Recreate the scheduler and link it to the NEW optimizer
mrpc_scheduler = get_linear_schedule_with_warmup(
    mrpc_optimizer,
    num_warmup_steps=mrpc_num_warmup_steps,
    num_training_steps=mrpc_num_training_steps
)

# 4. Load Metric
mrpc_metric = evaluate.load("glue", "mrpc")

In [ ]:
# Redefining the evaluation function to remove the hardcoded SST-2 metric
def evaluate_model_generic(model, dataloader, metric, device):
    model.eval()
    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        metric.add_batch(
            predictions=predictions.cpu(),
            references=batch["labels"].cpu()
        )
    return metric.compute()

# Run the training loop for MRPC!
for epoch in range(mrpc_epochs):
    avg_train_loss = train_one_epoch(mrpc_model, mrpc_train_dataloader, mrpc_optimizer, mrpc_scheduler, device)
    eval_result = evaluate_model_generic(mrpc_model, mrpc_eval_dataloader, mrpc_metric, device)

    print(f"Epoch {epoch + 1} (MRPC)")
    print(f"Average train loss: {avg_train_loss:.4f}")
    print(f"Eval result: {eval_result}")

E2E NLG & GPT-2

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

# 1. Load E2E NLG directly from the official GitHub CSVs
data_files = {
    "train": "https://github.com/tuetschek/e2e-dataset/raw/master/trainset.csv",
    "validation": "https://github.com/tuetschek/e2e-dataset/raw/master/devset.csv"
}
e2e_dataset = load_dataset("csv", data_files=data_files)

# 2. Load GPT-2 Tokenizer
gpt2_model_name = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_model_name)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

# 3. Tokenization for Causal LM
def tokenize_e2e(examples):
    # The CSV columns are named 'mr' and 'ref'
    inputs = [f"{mr} => {ref}{gpt2_tokenizer.eos_token}"
              for mr, ref in zip(examples['mr'], examples['ref'])]

    tokenized = gpt2_tokenizer(inputs, truncation=True, padding="max_length", max_length=128)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_e2e = e2e_dataset.map(tokenize_e2e, batched=True)
tokenized_e2e = tokenized_e2e.remove_columns(["mr", "ref"])
tokenized_e2e.set_format("torch")

# 4. Create DataLoaders
e2e_train_dataloader = DataLoader(tokenized_e2e["train"], shuffle=True, batch_size=8)
e2e_eval_dataloader = DataLoader(tokenized_e2e["validation"], shuffle=False, batch_size=8)

print("Successfully loaded and tokenized E2E dataset!")

In [ ]:
!pip install -q sacrebleu

In [ ]:
from transformers.pytorch_utils import Conv1D

class LoRAConv1D(nn.Module):
    """A LoRA wrapper designed specifically for Hugging Face's GPT-2 Conv1D layers."""
    def __init__(self, base_conv1d: Conv1D, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()

        # GPT-2's Conv1D weights are stored as (in_features, out_features)
        self.in_features = base_conv1d.weight.shape[0]
        self.out_features = base_conv1d.weight.shape[1]
        self.r = r
        self.scaling = alpha / r if r > 0 else 1.0

        # Keep original weights frozen
        self.weight = base_conv1d.weight
        self.weight.requires_grad = False

        if base_conv1d.bias is not None:
            self.bias = base_conv1d.bias
            self.bias.requires_grad = False
        else:
            self.bias = None

        self.dropout = nn.Dropout(dropout)

        if r > 0:
            self.lora_A = nn.Parameter(torch.zeros(self.in_features, r))
            self.lora_B = nn.Parameter(torch.zeros(r, self.out_features))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size_out = x.size()[:-1] + (self.out_features,)
        base_output = torch.addmm(self.bias, x.view(-1, x.size(-1)), self.weight)
        base_output = base_output.view(*size_out)

        if self.r > 0:
            lora_output = self.dropout(x) @ self.lora_A @ self.lora_B
            base_output += self.scaling * lora_output

        return base_output

In [ ]:
from transformers import AutoModelForCausalLM

# Load the model
gpt2_model = AutoModelForCausalLM.from_pretrained(gpt2_model_name)

# 1. Freeze all base parameters
for param in gpt2_model.parameters():
    param.requires_grad = False

# 2. Replace GPT-2 attention blocks (c_attn) with LoRAConv1D
for layer_idx, layer in enumerate(gpt2_model.transformer.h):
    old_c_attn = layer.attn.c_attn
    layer.attn.c_attn = LoRAConv1D(old_c_attn, r=4, alpha=32, dropout=0.1)

# 3. Send the entire model (including the new LoRA layers) to the GPU
gpt2_model.to(device)

print("GPT-2 LoRA Parameters:")
print_trainable_parameters(gpt2_model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 LoRA Parameters:
Trainable params: 147456
Total params:     124587264
Trainable ratio:  0.1184%


In [ ]:
# 1. Update Hyperparameters for GPT-2 based on the LoRA paper
gpt2_epochs = 5
gpt2_optimizer = torch.optim.AdamW(
    [p for p in gpt2_model.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=0.01
)

gpt2_num_training_steps = gpt2_epochs * len(e2e_train_dataloader)
gpt2_scheduler = get_linear_schedule_with_warmup(
    gpt2_optimizer,
    num_warmup_steps=500,
    num_training_steps=gpt2_num_training_steps
)

# Load the BLEU metric
bleu_metric = evaluate.load("sacrebleu")

def train_gpt2_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc="Training GPT-2", leave=False)

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)

def evaluate_gpt2_bleu(model, tokenizer, raw_val_dataset, metric, device, batch_size=8, limit=100):
    model.eval()
    predictions = []
    references = []

    tokenizer.padding_side = "left"

    eval_len = limit if limit is not None else len(raw_val_dataset)
    progress_bar = tqdm(range(0, eval_len, batch_size), desc="Generating Text for BLEU")

    for i in progress_bar:
        batch_mrs = raw_val_dataset["mr"][i:i+batch_size]
        batch_refs = raw_val_dataset["ref"][i:i+batch_size]

        prompts = [f"{mr} => " for mr in batch_mrs]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=50,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=False,
                num_beams=10,
                length_penalty=0.9,
            )

        decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for prompt, pred, ref in zip(prompts, decoded_preds, batch_refs):
            generated_text = pred[len(prompt):].strip()
            predictions.append(generated_text)
            references.append([ref])

    # Switch back to right-padding just in case you run training again
    tokenizer.padding_side = "right"

    results = metric.compute(predictions=predictions, references=references)
    return results, predictions, references

# 2. Add the loop to run for the full 5 epochs
for epoch in range(gpt2_epochs):
    avg_loss = train_gpt2_epoch(gpt2_model, e2e_train_dataloader, gpt2_optimizer, gpt2_scheduler, device)
    print(f"Epoch {epoch + 1} | GPT-2 Average Train Loss: {avg_loss:.4f}")

    # Run the evaluation
    bleu_results, sample_preds, sample_refs = evaluate_gpt2_bleu(
        gpt2_model,
        gpt2_tokenizer,
        e2e_dataset["validation"],
        bleu_metric,
        device,
        batch_size=8,
        limit=160
    )

    print(f"\nGPT-2 LoRA BLEU Score: {bleu_results['score']:.2f}")

In [ ]:
# 1. Update Hyperparameters for GPT-2 based on the LoRA paper
gpt2_epochs = 5
gpt2_optimizer = torch.optim.AdamW(
    [p for p in gpt2_model.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=0.01
)

gpt2_num_training_steps = gpt2_epochs * len(e2e_train_dataloader)
gpt2_scheduler = get_linear_schedule_with_warmup(
    gpt2_optimizer,
    num_warmup_steps=500,
    num_training_steps=gpt2_num_training_steps
)

# Load the BLEU metric
bleu_metric = evaluate.load("sacrebleu")

def train_gpt2_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc="Training GPT-2", leave=False)

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)

def group_references(raw_val_dataset):
    """Group all references by their MR."""
    mr_to_refs = defaultdict(list)
    for mr, ref in zip(raw_val_dataset["mr"], raw_val_dataset["ref"]):
        mr_to_refs[mr].append(ref)
    # Return as list of (mr, [refs]) preserving insertion order
    seen = {}
    for mr in raw_val_dataset["mr"]:
        if mr not in seen:
            seen[mr] = mr_to_refs[mr]
    return list(seen.keys()), list(seen.values())

def evaluate_gpt2_bleu(model, tokenizer, raw_val_dataset, device, batch_size=8, limit=100):
    model.eval()
    predictions = []
    references = []  # will be list of lists: [[ref1, ref2, ...], ...]

    tokenizer.padding_side = "left"

    # Group refs by MR first
    unique_mrs, grouped_refs = group_references(raw_val_dataset)

    eval_len = min(limit, len(unique_mrs)) if limit else len(unique_mrs)
    unique_mrs = unique_mrs[:eval_len]
    grouped_refs = grouped_refs[:eval_len]

    progress_bar = tqdm(range(0, eval_len, batch_size), desc="Generating Text for BLEU")

    for i in progress_bar:
        batch_mrs = unique_mrs[i:i+batch_size]
        batch_refs = grouped_refs[i:i+batch_size]  # list of lists

        prompts = [f"{mr} => " for mr in batch_mrs]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=50,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=False,
                num_beams=10,
                length_penalty=0.9,
            )

        decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for prompt, pred, refs in zip(prompts, decoded_preds, batch_refs):
            generated_text = pred[len(prompt):].strip()
            predictions.append(generated_text)
            references.append(refs)  # full list of refs for this MR

    tokenizer.padding_side = "right"

    # sacrebleu multi-reference format
    bleu = BLEU()
    max_refs = max(len(r) for r in references)
    ref_streams = []
    for i in range(max_refs):
        ref_streams.append([
            refs[i] if i < len(refs) else None
            for refs in references
        ])

    result = bleu.corpus_score(predictions, ref_streams)
    return result, predictions, references

# 2. Add the loop to run for the full 5 epochs
for epoch in range(gpt2_epochs):
    avg_loss = train_gpt2_epoch(gpt2_model, e2e_train_dataloader, gpt2_optimizer, gpt2_scheduler, device)
    print(f"Epoch {epoch + 1} | GPT-2 Average Train Loss: {avg_loss:.4f}")

    # Run the evaluation
    bleu_results, sample_preds, sample_refs = evaluate_gpt2_bleu(
        gpt2_model,
        gpt2_tokenizer,
        e2e_dataset["validation"],
        device,
        batch_size=8,
        limit=160
    )

    print(f"\nGPT-2 LoRA BLEU Score: {bleu_results}")

Training GPT-2:   0%|          | 0/5258 [00:00<?, ?it/s]

Epoch 1 | GPT-2 Average Train Loss: 0.5895


Generating Text for BLEU:   0%|          | 0/20 [00:00<?, ?it/s]


GPT-2 LoRA BLEU Score: BLEU = 65.81 92.1/75.9/59.5/45.1 (BP = 1.000 ratio = 1.001 hyp_len = 3993 ref_len = 3989)


Training GPT-2:   0%|          | 0/5258 [00:00<?, ?it/s]

...


GPT-2 LoRA BLEU Score: BLEU = 64.59 89.4/73.9/58.3/45.2 (BP = 1.000 ratio = 1.027 hyp_len = 4481 ref_len = 4362)


Training GPT-2:   0%|          | 0/5258 [00:00<?, ?it/s]

Epoch 5 | GPT-2 Average Train Loss: 0.3599


Generating Text for BLEU:   0%|          | 0/20 [00:00<?, ?it/s]


GPT-2 LoRA BLEU Score: BLEU = 65.88 90.4/74.8/59.3/47.0 (BP = 1.000 ratio = 1.023 hyp_len = 4406 ref_len = 4309)
